In [ ]:
# Cell 1: Install dependencies
!pip install fastapi uvicorn pyngrok requests transformers underthesea -q
!pip install torch --index-url https://download.pytorch.org/whl/cu118 -q 2>/dev/null || pip install torch -q


In [ ]:
# Cell 2: Upload worker files
# 1. Upload app.py, crawler.py, inferencer.py, scoring.py to /content/
# 2. Create folder /content/phobert_model/ and upload:
#    model.safetensors, config.json, tokenizer_config.json, vocab.txt, bpe.codes, added_tokens.json
import os
os.makedirs("/content/phobert_model", exist_ok=True)
os.chdir("/content")
print("Working directory:", os.getcwd())
print("phobert_model folder ready:", os.path.exists("/content/phobert_model"))

In [ ]:
# Cell 3: Config — update these values
WORKER_API_KEY = 'worker-secret'  # must match backend WORKER_API_KEY env var
MAIN_API_URL = 'https://YOUR-RAILWAY-APP.railway.app'  # update after Railway deploy
NGROK_TOKEN = ''  # get free token from ngrok.com dashboard

import os
os.environ['WORKER_API_KEY'] = WORKER_API_KEY
print('Config set.')

In [ ]:
# Cell 4: Start FastAPI server in background thread
import threading
import uvicorn
from app import app

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8001, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(2)
print('Worker server started on port 8001')

In [ ]:
# Cell 5: Create ngrok tunnel + register with Main API
from pyngrok import ngrok, conf
import requests

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN

tunnel = ngrok.connect(8001)
worker_url = tunnel.public_url
print(f'Worker URL: {worker_url}')

# Register with Main API
r = requests.post(
    f'{MAIN_API_URL}/worker/register',
    json={'url': worker_url},
    headers={'x-worker-key': WORKER_API_KEY},
    timeout=10,
)
print(f'Registration: {r.status_code} {r.json()}')

In [ ]:
# Cell 6: Keep alive — run this cell to prevent Colab timeout
import time
print('Worker is running. Keep this cell alive to stay online.')
while True:
    time.sleep(60)
    print('... still alive')